# 🔍 BOSS Datalake — Script 1: Source (Postgres) vs Bronze (Trino)

### Checks
| # | Check |
|---|-------|
| 1 | Column Count — Source vs Bronze |
| 2 | Data Type Validation |
| 3 | Row Count (retention filter + duplicate count) |
| 4 | Pipeline Columns Null Check |
| 5 | NULL % + Garbage Check (column-wise) |
| 6 | True NULL vs False NULL (Id join) |

---
## 📦 Cell 1 — Libraries

In [53]:
import re
import pandas as pd
import urllib3
from decimal import Decimal, InvalidOperation, getcontext
from IPython.display import display
from trino.dbapi import connect as trino_connect
from trino.auth import BasicAuthentication
import psycopg2

getcontext().prec = 38
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)
print('✅ Libraries loaded!')

✅ Libraries loaded!


---
## ⚙️ Cell 2 — Configuration

In [60]:
# ── PostgreSQL ──
PG_CONFIG = {
    'host'    : 'boss-archive-db.cf2ctiswz4if.us-east-1.rds.amazonaws.com',
    'port'    : 5432,
    'dbname'  : 'AlpineUATvClearing_Archive',
    'user'    : 'alpine_readonly_user',
    'password': 'd7=XZ3mb4oHU',
    'sslmode' : 'require' 
}
# ── Table config (boss_metadata se lo) ──
SOURCE_TABLE   = 'StreetComparison'
RETENTION_DAYS = 160
PARTITION_COL  = 'SystemDate'
PRIMARY_KEY    = ['Id']   # single: 'Id' | composite: ['Id', 'SystemDate']

# ── Trino ──
TRINO_CONFIG = {
    'host'    : '3.221.185.123',
    'port'    : 8443,
    'user'    : 'shariq.mehmood',
    'password': 'Prometheus-X!',
    'catalog' : 'iceberg',
    'schema'  : 'qa'
}
BRONZE_TABLE = 'iceberg.qa.bronze_StreetComparison'

# ── Pipeline columns ──
PIPELINE_COLUMNS = []

# ── Sample size for NULL validation ──
NULL_SAMPLE_SIZE = 1000

# ── PK helper ──
def get_pk_expression(pk):
    if isinstance(pk, list):
        parts = [f'CAST("{p}" AS VARCHAR)' for p in pk]
        return " || '_' || ".join(parts)
    return f'CAST("{pk}" AS VARCHAR)'

PK_COLS = PRIMARY_KEY if isinstance(PRIMARY_KEY, list) else [PRIMARY_KEY]
PK_EXPR = get_pk_expression(PRIMARY_KEY)

print(f'✅ Source   : {SOURCE_TABLE}')
print(f'✅ Bronze   : {BRONZE_TABLE}')
print(f'✅ Retention: {RETENTION_DAYS} days on {PARTITION_COL}')
print(f'✅ PK       : {PRIMARY_KEY}')
print(f'✅ PK expr  : {PK_EXPR}')
print(f'✅ Pipeline : {PIPELINE_COLUMNS}')

✅ Source   : StreetComparison
✅ Bronze   : iceberg.qa.bronze_StreetComparison
✅ Retention: 160 days on SystemDate
✅ PK       : ['Id']
✅ PK expr  : CAST("Id" AS VARCHAR)
✅ Pipeline : []


---
## 🔌 Cell 3 — Connections

In [55]:
def get_pg_connection():
    return psycopg2.connect(**PG_CONFIG)

def get_trino_connection():
    auth = BasicAuthentication(TRINO_CONFIG['user'], TRINO_CONFIG['password'])
    return trino_connect(
        host=TRINO_CONFIG['host'], port=TRINO_CONFIG['port'],
        user=TRINO_CONFIG['user'], auth=auth,
        http_scheme='https', verify=False,
        catalog=TRINO_CONFIG['catalog'], schema=TRINO_CONFIG['schema']
    )

try:
    pg_conn = get_pg_connection()
    print('✅ PostgreSQL connected!')
    pg_conn.close()
except Exception as e:
    print(f'❌ PostgreSQL failed: {e}')

try:
    tr_conn = get_trino_connection()
    cur = tr_conn.cursor()
    cur.execute('SELECT 1')
    cur.fetchall()
    print('✅ Trino connected!')
except Exception as e:
    print(f'❌ Trino failed: {e}')

✅ PostgreSQL connected!
✅ Trino connected!


---
## 📋 Cell 4 — Column Count Check

In [56]:
def get_source_columns():
    pg_conn = get_pg_connection()
    df = pd.read_sql("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = %s
        ORDER BY ordinal_position
    """, pg_conn, params=(SOURCE_TABLE,))
    pg_conn.close()
    df['column_name'] = df['column_name'].str.lower()
    return df

def get_bronze_columns():
    tr_conn = get_trino_connection()
    cur = tr_conn.cursor()
    cur.execute(f'DESCRIBE {BRONZE_TABLE}')
    rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=['column_name','data_type','extra','comment'])
    df['column_name'] = df['column_name'].str.lower()
    return df[['column_name','data_type']]

src_cols_df    = get_source_columns()
bronze_cols_df = get_bronze_columns()

src_set      = set(src_cols_df['column_name'])
pipeline_set = set(c.lower() for c in PIPELINE_COLUMNS)
bronze_set   = set(bronze_cols_df['column_name'])
bronze_excl  = bronze_set - pipeline_set

matched           = src_set & bronze_excl
missing_in_bronze = src_set - bronze_excl
extra_in_bronze   = bronze_excl - src_set

print('=' * 65)
print('📋 COLUMN COUNT CHECK — SOURCE vs BRONZE')
print('=' * 65)
print(f'  Source columns              : {len(src_set)}')
print(f'  Bronze columns (total)      : {len(bronze_set)}')
print(f'  Pipeline columns (excluded) : {len(pipeline_set)}')
print(f'  Bronze columns (excl pipe)  : {len(bronze_excl)}')
print(f'  ✅ Matched                  : {len(matched)}')
print(f'\n🔴 Missing in Bronze ({len(missing_in_bronze)}):')
for c in sorted(missing_in_bronze): print(f'  -> {c}')
if not missing_in_bronze: print('  ✅ None')
print(f'\n🔵 Extra in Bronze ({len(extra_in_bronze)}):')
for c in sorted(extra_in_bronze): print(f'  -> {c}')
if not extra_in_bronze: print('  ✅ None')

/tmp/ipykernel_15121/3395541006.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


📋 COLUMN COUNT CHECK — SOURCE vs BRONZE
  Source columns              : 40
  Bronze columns (total)      : 42
  Pipeline columns (excluded) : 0
  Bronze columns (excl pipe)  : 42
  ✅ Matched                  : 40

🔴 Missing in Bronze (0):
  ✅ None

🔵 Extra in Bronze (2):
  -> ingested_ts
  -> source_table


---
## 🧪 Cell 5 — Data Type Validation

In [57]:
PG_TYPE_MAP = {
    'integer': 'integer', 'bigint': 'bigint', 'smallint': 'smallint',
    'numeric': 'decimal', 'decimal': 'decimal',
    'double precision': 'double', 'real': 'double',
    'character varying': 'varchar', 'character': 'varchar',
    'text': 'varchar', 'boolean': 'boolean', 'date': 'date',
    'timestamp without time zone': 'timestamp',
    'timestamp with time zone': 'timestamp',
    'uuid': 'varchar', 'json': 'varchar', 'jsonb': 'varchar',
}
TRINO_TYPE_MAP = {
    'integer': 'integer', 'bigint': 'bigint', 'smallint': 'smallint',
    'decimal': 'decimal', 'double': 'double', 'varchar': 'varchar',
    'boolean': 'boolean', 'date': 'date', 'timestamp': 'timestamp',
}

def norm_pg(t):    return PG_TYPE_MAP.get(str(t).lower().strip(), str(t).lower().strip())
def norm_trino(t): return TRINO_TYPE_MAP.get(str(t).lower().split('(')[0].strip(), str(t).lower().split('(')[0].strip())

src_type_map    = dict(zip(src_cols_df['column_name'], src_cols_df['data_type']))
bronze_type_map = dict(zip(bronze_cols_df['column_name'], bronze_cols_df['data_type']))

print('=' * 85)
print('🧪 DATA TYPE VALIDATION — SOURCE vs BRONZE')
print('=' * 85)
print(f'  {"COLUMN":<35} | {"SOURCE TYPE":<22} | {"BRONZE TYPE":<22} | STATUS')
print(f'  {"-" * 85}')

dtype_results = []
for col in sorted(matched):
    s_raw  = src_type_map.get(col, 'unknown')
    b_raw  = bronze_type_map.get(col, 'unknown')
    ok     = norm_pg(s_raw) == norm_trino(b_raw)
    status = '✅ MATCH' if ok else '❌ MISMATCH'
    print(f'  {col:<35} | {s_raw:<22} | {b_raw:<22} | {status}')
    dtype_results.append({'column': col, 'source_type': s_raw, 'bronze_type': b_raw, 'status': status})

dtype_df = pd.DataFrame(dtype_results)
mm = len(dtype_df[dtype_df['status'] == '❌ MISMATCH'])
print(f'  {"-" * 85}')
print(f'\n  ✅ Match   : {len(dtype_df)-mm}/{len(dtype_df)}')
print(f'  ❌ Mismatch: {mm}/{len(dtype_df)}')

🧪 DATA TYPE VALIDATION — SOURCE vs BRONZE
  COLUMN                              | SOURCE TYPE            | BRONZE TYPE            | STATUS
  -------------------------------------------------------------------------------------
  accountnumber                       | character varying      | varchar                | ✅ MATCH
  accounttype                         | character varying      | varchar                | ✅ MATCH
  assettype                           | character varying      | varchar                | ✅ MATCH
  breakamount                         | numeric                | decimal(19,6)          | ✅ MATCH
  breakqty                            | numeric                | decimal(19,8)          | ✅ MATCH
  breaks                              | character varying      | varchar                | ✅ MATCH
  comp                                | character varying      | varchar                | ✅ MATCH
  comparedate                         | timestamp without time zone | timestamp(6)     

---
## 🔢 Cell 6 — Row Count Check

In [61]:
def get_source_count():
    pg_conn = get_pg_connection()
    cur = pg_conn.cursor()
    cur.execute(f'SELECT COUNT(*) FROM "{SOURCE_TABLE}"')
    count = cur.fetchone()[0]
    pg_conn.close()
    return count

def get_bronze_counts():
    tr_conn = get_trino_connection()
    cur = tr_conn.cursor()
    cur.execute(f'SELECT COUNT(*), COUNT(DISTINCT {PK_EXPR}) FROM {BRONZE_TABLE}')
    row = cur.fetchone()
    return row[0], row[1]

src_count              = get_source_count()
bronze_total, bronze_unique = get_bronze_counts()
bronze_dups            = bronze_total - bronze_unique
match                  = src_count == bronze_unique

print('=' * 65)
print('🔢 ROW COUNT CHECK — SOURCE vs BRONZE')
print('=' * 65)
print(f'  Source rows (full table)       : {src_count}')
print(f'  Bronze rows (total)            : {bronze_total}')
print(f'  Bronze duplicates              : {bronze_dups}')
print(f'  Bronze unique rows             : {bronze_unique}')
print(f'\n  Source == Bronze unique        : {"✅ MATCH" if match else "❌ MISMATCH"}')
if not match:
    print(f'  Difference                     : {bronze_unique - src_count:+d}')

🔢 ROW COUNT CHECK — SOURCE vs BRONZE
  Source rows (full table)       : 560023109
  Bronze rows (total)            : 1120046218
  Bronze duplicates              : 560023143
  Bronze unique rows             : 560023075

  Source == Bronze unique        : ❌ MISMATCH
  Difference                     : -34


---
## 🔬 Cell 8 — NULL % + Garbage Check (Bronze column-wise)

In [ ]:
def is_garbage(value):
    if not value or str(value).strip() == '':
        return False
    return bool(re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]').search(str(value)))

def fetch_bronze_sample():
    tr_conn = get_trino_connection()
    cur = tr_conn.cursor()
    cur.execute(f'SELECT * FROM {BRONZE_TABLE} LIMIT {NULL_SAMPLE_SIZE}')
    rows = cur.fetchall()
    cols = [d[0].lower() for d in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    print(f'✅ Bronze sample: {len(df)} rows fetched')
    return df

bronze_sample_df = fetch_bronze_sample()
check_cols       = [c for c in bronze_sample_df.columns if c not in pipeline_set]
total_rows       = len(bronze_sample_df)

print('\n' + '=' * 90)
print('🔬 NULL % + GARBAGE CHECK — BRONZE')
print('   (Sirf columns with NULLs ya Garbage print honge)')
print('=' * 90)
print(f'  {"COLUMN":<35} | {"NULL COUNT":>10} | {"NULL %":>8} | {"GARBAGE":>8} | STATUS')
print(f'  {"-" * 85}')

null_garbage_results = []
for col in sorted(check_cols):
    vals          = bronze_sample_df[col].tolist()
    null_count    = sum(1 for v in vals if pd.isna(v) or str(v).strip() in ['','None','null','nan'])
    garbage_count = sum(1 for v in vals if not pd.isna(v) and is_garbage(str(v)))
    null_pct      = round((null_count / total_rows) * 100, 2) if total_rows > 0 else 0

    if garbage_count > 0:
        status = '❌ GARBAGE'
    elif null_pct >= 50:
        status = f'⚠️  HIGH NULL ({null_pct}%)'
    elif null_count > 0:
        status = f'⚠️  NULLS ({null_pct}%)'
    else:
        status = '✅ PASS'

    if null_count > 0 or garbage_count > 0:
        print(f'  {col:<35} | {null_count:>10} | {null_pct:>7}% | {garbage_count:>8} | {status}')

    null_garbage_results.append({
        'column': col, 'null_count': null_count,
        'null_pct': null_pct, 'garbage_count': garbage_count, 'status': status
    })

ng_df = pd.DataFrame(null_garbage_results)
print(f'  {"-" * 85}')
print(f'  Columns with NULLs   : {len(ng_df[ng_df["null_count"] > 0])}')
print(f'  Columns with Garbage : {len(ng_df[ng_df["garbage_count"] > 0])}')
print(f'  Clean columns        : {len(ng_df[ng_df["status"] == "✅ PASS"])}')

---
## ✅ Cell 9 — True NULL vs False NULL (Id join)

In [ ]:
def fetch_source_sample():
    pg_conn = get_pg_connection()
    df = pd.read_sql(f"""
        SELECT * FROM "{SOURCE_TABLE}"
        LIMIT {NULL_SAMPLE_SIZE}
    """, pg_conn)
    pg_conn.close()
    df.columns = df.columns.str.lower()
    print(f'✅ Source sample : {len(df)} rows')
    return df

def fetch_bronze_by_ids(id_list):
    tr_conn = get_trino_connection()
    cur = tr_conn.cursor()
    ids_str = ', '.join(str(i) for i in id_list)
    cur.execute(f'SELECT * FROM {BRONZE_TABLE} WHERE "{PK_EXPR}" IN ({ids_str})')
    rows = cur.fetchall()
    cols = [d[0].lower() for d in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    # Dedup — latest ingestedfordate per Id
    if 'ingestedfordate' in df.columns:
        df = df.sort_values('ingestedfordate', ascending=False)
        df = df.drop_duplicates(subset=[PK_EXPR.lower()], keep='first')
    print(f'✅ Bronze sample : {len(df)} rows (deduped by Id)')
    return df

src_sample    = fetch_source_sample()
id_list       = src_sample[PK_EXPR.lower()].tolist()
bronze_sample = fetch_bronze_by_ids(id_list)

pk = PK_EXPR.lower()
src_sample[pk]    = src_sample[pk].astype(str).str.strip()
bronze_sample[pk] = bronze_sample[pk].astype(str).str.strip()
merged = src_sample.merge(bronze_sample, on=pk, suffixes=('_src','_bronze'))

print(f'\n   Source rows  : {len(src_sample)}')
print(f'   Bronze rows  : {len(bronze_sample)}')
print(f'   Joined rows  : {len(merged)}')
if len(src_sample) - len(merged) > 0:
    print(f'   ⚠️  Unmatched : {len(src_sample)-len(merged)} (Id Bronze mein nahi mili)')

skip      = pipeline_set | {pk}
chk_cols  = [c.replace('_src','') for c in merged.columns
             if c.endswith('_src') and c.replace('_src','') not in skip]

print('\n' + '=' * 95)
print('✅ TRUE NULL vs FALSE NULL — SOURCE vs BRONZE (Id join)')
print('  ✅ TRUE NULL  = source mein bhi NULL tha (expected)')
print('  ❌ FALSE NULL = source mein value thi, Bronze mein NULL (ingestion bug)')
print('  ⏭️  SKIP       = Bronze mein koi NULL nahi')
print('=' * 95)
print(f'  {"COLUMN":<35} | {"BRONZE NULL":>11} | {"SRC NULL":>8} | {"FALSE NULL":>10} | STATUS')
print(f'  {"-" * 90}')

true_null_results = []
false_null_detail = []

for col in sorted(chk_cols):
    sc = col + '_src'
    bc = col + '_bronze'
    if sc not in merged.columns or bc not in merged.columns:
        continue

    t_vals = merged[bc]
    s_vals = merged[sc]

    bronze_null = t_vals.isna() | t_vals.astype(str).str.strip().isin(['','None','null','nan','NaN'])
    src_null    = s_vals.isna() | s_vals.astype(str).str.strip().isin(['','None','null','nan','NaN'])
    false_null  = bronze_null & ~src_null

    bn = int(bronze_null.sum())
    sn = int(src_null.sum())
    fn = int(false_null.sum())

    if bn == 0:
        status = '⏭️  SKIP'
    elif fn > 0:
        status = '❌ FALSE NULL'
        false_rows = merged[false_null][[pk, sc]].copy()
        if fn <= 5:
            for _, r in false_rows.iterrows():
                false_null_detail.append({
                    'column': col,
                    PK_EXPR: r[pk],
                    'source_value': str(r[sc])[:50]
                })
        else:
            sample_ids = false_rows[pk].head(5).tolist()
            false_null_detail.append({
                'column': col,
                PK_EXPR: f'{fn} rows — sample: {sample_ids}',
                'source_value': 'multiple'
            })
    else:
        status = '✅ TRUE NULL'

    if bn > 0:
        print(f'  {col:<35} | {bn:>11} | {sn:>8} | {fn:>10} | {status}')

    true_null_results.append({
        'column': col, 'bronze_null': bn,
        'src_null': sn, 'false_null': fn, 'status': status
    })

print(f'  {"-" * 90}')

if false_null_detail:
    print(f'\n❌ FALSE NULL DETAIL:')
    print(f'  {"COLUMN":<35} | {PK_EXPR:<40} | SOURCE VALUE')
    print(f'  {"-" * 90}')
    for r in false_null_detail:
        print(f'  {str(r["column"]):<35} | {str(r[PK_EXPR]):<40} | {str(r["source_value"])[:30]}')
else:
    print('\n✅ Koi FALSE NULL nahi — sab nulls source mein bhi the!')

true_null_df  = pd.DataFrame(true_null_results)
false_null_df = pd.DataFrame(false_null_detail) if false_null_detail else pd.DataFrame()

---
## 📊 Cell 10 — Final Summary

In [52]:
col_ok      = len(missing_in_bronze) == 0 and len(extra_in_bronze) == 0
dtype_ok    = len(dtype_df[dtype_df['status'] == '❌ MISMATCH']) == 0
row_ok      = src_count == bronze_unique
garbage_ok  = len(ng_df[ng_df['garbage_count'] > 0]) == 0
false_ok    = len(false_null_df) == 0

checks = [
    ('Column Count',     '✅ PASS' if col_ok      else '❌ FAIL',
     f'Missing: {len(missing_in_bronze)}, Extra: {len(extra_in_bronze)}'),
    ('Data Types',       '✅ PASS' if dtype_ok    else '❌ FAIL',
     f'Mismatches: {len(dtype_df[dtype_df["status"]=="❌ MISMATCH"])}'),
    ('Row Count',        '✅ PASS' if row_ok      else '❌ FAIL',
     f'Source: {src_count}, Bronze unique: {bronze_unique}, Dups: {bronze_dups}'),
    ('Garbage Values',   '✅ PASS' if garbage_ok  else '❌ FAIL',
     f'Columns with garbage: {len(ng_df[ng_df["garbage_count"] > 0])}'),
    ('True NULL Check',  '✅ PASS' if false_ok    else '❌ FAIL',
     f'False nulls: {len(false_null_df)} columns'),
]

print('=' * 75)
print('📊 FINAL SUMMARY — SOURCE vs BRONZE')
print('=' * 75)
print(f'  {"CHECK":<25} | {"STATUS":<12} | DETAIL')
print(f'  {"-" * 70}')
for check, status, detail in checks:
    print(f'  {check:<25} | {status:<12} | {detail}')
print(f'  {"-" * 70}')
print(f'  Source : {SOURCE_TABLE}')
print(f'  Bronze : {BRONZE_TABLE}')
print('=' * 75)
overall = all(s == '✅ PASS' for _, s, _ in checks)
print('\n🎉 ALL CHECKS PASSED!' if overall else '\n❌ SOME CHECKS FAILED — Details upar dekho!')

📊 FINAL SUMMARY — SOURCE vs BRONZE
  CHECK                     | STATUS       | DETAIL
  ----------------------------------------------------------------------
  Column Count              | ❌ FAIL       | Missing: 0, Extra: 2
  Data Types                | ✅ PASS       | Mismatches: 0
  Row Count                 | ✅ PASS       | Source: 998114539, Bronze unique: 998114539, Dups: 998114539
  Garbage Values            | ✅ PASS       | Columns with garbage: 0
  True NULL Check           | ✅ PASS       | False nulls: 0 columns
  ----------------------------------------------------------------------
  Source : transaction
  Bronze : iceberg.qa.bronze_transaction

❌ SOME CHECKS FAILED — Details upar dekho!
